<a href="https://colab.research.google.com/github/Abhishek05-mavrick/AI/blob/main/AdaBoost_Classifier_with_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# NOTEBOOK.
import kagglehub
uciml_pima_indians_diabetes_database_path = kagglehub.dataset_download('uciml/pima-indians-diabetes-database')

print('Data source import complete.')


Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.
Data source import complete.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [4]:
df = pd.read_csv(f"{uciml_pima_indians_diabetes_database_path}/diabetes.csv")
print(df.head())
print(df.info())
print(df.describe())  # columns like BMI, BloodPressure, SkinThickness cannot be 0 — needs cleaning
print(df.shape)

print(df["Insulin"].value_counts())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768

In [5]:
columns_check = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

for col in columns_check:
    zero_counts      = (df[col] == 0).sum()
    zero_percentage  = 100 * zero_counts / len(df[col])
    print(f"{col} — zero count: {zero_counts} || {zero_percentage:.2f}%")
    # BloodPressure, SkinThickness, Insulin, BMI have significant zero counts
    # dropping these rows would lose too much data — median imputation chosen instead

Glucose — zero count: 5 || 0.65%
BloodPressure — zero count: 35 || 4.56%
SkinThickness — zero count: 227 || 29.56%
Insulin — zero count: 374 || 48.70%
BMI — zero count: 11 || 1.43%


In [6]:
# split before cleaning to prevent data leakage
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=15)

# compute median from non-zero train values only — store for reuse on test set
medians = {}
for col in columns_check:
    median_value  = X_train[X_train[col] != 0][col].median()  # median of non-zero rows in X_train
    medians[col]  = median_value
    X_train[col]  = X_train[col].replace(0, median_value)

# apply train medians to test set — no refit
for col in columns_check:
    X_test[col] = X_test[col].replace(0, medians[col])

print(X_train.describe())  # min values should now be non-zero

       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   614.000000  614.000000     614.000000     614.000000  614.000000   
mean      3.907166  121.560261      72.612378      29.040717  142.477199   
std       3.385438   29.974412      12.165642       8.312217   80.879330   
min       0.000000   44.000000      24.000000       7.000000   14.000000   
25%       1.000000  100.000000      64.000000      25.000000  125.000000   
50%       3.000000  117.000000      72.000000      29.000000  129.500000   
75%       6.000000  139.750000      80.000000      32.000000  130.000000   
max      17.000000  199.000000     122.000000      63.000000  680.000000   

              BMI  DiabetesPedigreeFunction         Age  
count  614.000000                614.000000  614.000000  
mean    32.448208                  0.469948   33.285016  
std      6.862948                  0.328516   11.678337  
min     18.200000                  0.084000   21.000000  
25%     27.600000        

---
## 5. Baseline Model — AdaBoost

In [7]:
ada = AdaBoostClassifier()
ada.fit(X_train, y_train)
y_pred = ada.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Accuracy Score:", accuracy_score(y_test, y_pred))

Confusion Matrix:
 [[87 21]
 [17 29]]
Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.81      0.82       108
           1       0.58      0.63      0.60        46

    accuracy                           0.75       154
   macro avg       0.71      0.72      0.71       154
weighted avg       0.76      0.75      0.76       154

Accuracy Score: 0.7532467532467533


In [8]:
param_grid = {
    "n_estimators" : [50, 60, 80, 200, 500],
    "learning_rate": [0.001, 0.01, 0.1, 1.0, 1.5, 2.0, 2.5, 7]
}

grid = GridSearchCV(estimator=ada, param_grid=param_grid, cv=3, verbose=1, n_jobs=-1)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)

Fitting 3 folds for each of 40 candidates, totalling 120 fits
Best params: {'learning_rate': 1.5, 'n_estimators': 200}


In [9]:
ada2 = AdaBoostClassifier(learning_rate=1.5, n_estimators=200)
ada2.fit(X_train, y_train)
y_pred_ada2 = ada2.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ada2))
print("Classification Report:\n", classification_report(y_test, y_pred_ada2))
print("Accuracy Score:", accuracy_score(y_test, y_pred_ada2))
# accuracy dropped to ~0.73 after tuning — GridSearchCV result did not generalize better in this case suprisingly

Confusion Matrix:
 [[85 23]
 [19 27]]
Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.79      0.80       108
           1       0.54      0.59      0.56        46

    accuracy                           0.73       154
   macro avg       0.68      0.69      0.68       154
weighted avg       0.73      0.73      0.73       154

Accuracy Score: 0.7272727272727273
